# Flatworm Simulator: Chemotaxis, Feeding, Fission & Starvation

This notebook simulates planarian worms on a toroidal grid. Worms move by heading, drain energy over time, feed on nearby food pellets, split when energy is high, and die when energy hits zero.

**Hybrid pattern**: Rust reducers handle the hot-loop physics (movement and metabolism), while Python handles structural decisions (feeding via spatial lookup, fission spawning, starvation despawn) because these require cross-archetype queries and spawn/despawn operations.

**Setup:**
```bash
cd crates/minkowski-py
uv sync --all-extras && source .venv/bin/activate
maturin develop --release
```

In [ ]:
import random

import matplotlib.pyplot as plt
import minkowski_py as mk
from IPython.display import clear_output

N_WORMS = 200
N_FOOD = 500
WORLD_SIZE = 500.0
FRAMES = 300
FEED_RADIUS = 20.0

## Spawn Worms and Food

Worms have `Energy, Heading, Position, WormSize`. Food has `Nutrition, Position`. The ECS distinguishes them by component presence — no marker components needed.

In [ ]:
world = mk.World()
random.seed(42)

world.spawn_batch(
    "Energy,Heading,Position,WormSize",
    {
        "energy": [100.0] * N_WORMS,
        "heading": [random.uniform(0, 6.28) for _ in range(N_WORMS)],
        "pos_x": [random.uniform(0, WORLD_SIZE) for _ in range(N_WORMS)],
        "pos_y": [random.uniform(0, WORLD_SIZE) for _ in range(N_WORMS)],
        "worm_size": [1.5] * N_WORMS,
    },
)

world.spawn_batch(
    "Nutrition,Position",
    {
        "nutrition": [50.0] * N_FOOD,
        "pos_x": [random.uniform(0, WORLD_SIZE) for _ in range(N_FOOD)],
        "pos_y": [random.uniform(0, WORLD_SIZE) for _ in range(N_FOOD)],
    },
)

registry = mk.ReducerRegistry(world)
grid = mk.SpatialGrid(world_size=WORLD_SIZE, cell_size=FEED_RADIUS)

print(f"Spawned {N_WORMS} worms + {N_FOOD} food")

## Simulation Loop

**Rust** handles movement + metabolism (per-entity physics).  
**Python** handles feeding (cross-archetype spatial lookup), fission (spawn), and starvation (despawn).

In [ ]:
worm_counts = []
food_counts = []
frames = []

for frame in range(FRAMES):
    # ── Rust: physics ──
    registry.run("worm_move", world, dt=1.0, world_size=WORLD_SIZE, speed=2.0)
    registry.run("worm_metabolism", world, dt=1.0, drain_rate=0.3)

    # ── Python: feeding via spatial lookup ──
    grid.rebuild(world)
    worms = world.query("Energy", "Position", "WormSize")
    food = world.query("Nutrition", "Position")

    if len(food) > 0 and len(worms) > 0:
        food_id_set = set(food["entity_id"].to_list())
        eaten = set()
        fed_worm_ids = []

        for i in range(len(worms)):
            neighbors = grid.query_radius(
                worms["pos_x"][i], worms["pos_y"][i], FEED_RADIUS
            )
            for nid in neighbors:
                if nid in food_id_set and nid not in eaten:
                    eaten.add(nid)
                    fed_worm_ids.append(worms["entity_id"][i])
                    break

        for fid in eaten:
            world.despawn(fid)
        if fed_worm_ids:
            worms2 = world.query("Energy", "WormSize")
            id_to_idx = {worms2["entity_id"][i]: i for i in range(len(worms2))}
            boost_ids = [wid for wid in fed_worm_ids if wid in id_to_idx]
            if boost_ids:
                new_energy = [
                    min(worms2["energy"][id_to_idx[wid]] + 30.0, 200.0)
                    for wid in boost_ids
                ]
                new_size = [
                    min(worms2["worm_size"][id_to_idx[wid]] + 0.3, 5.0)
                    for wid in boost_ids
                ]
                world.write_column("Energy", boost_ids, energy=new_energy)
                world.write_column("WormSize", boost_ids, worm_size=new_size)

    # ── Python: fission ──
    worms = world.query("Energy", "Heading", "Position", "WormSize")
    for i in range(len(worms)):
        if worms["energy"][i] > 150.0 and worms["worm_size"][i] > 2.5:
            pid = worms["entity_id"][i]
            new_e = worms["energy"][i] * 0.5
            new_s = worms["worm_size"][i] * 0.6
            world.write_column("Energy", [pid], energy=[new_e])
            world.write_column("WormSize", [pid], worm_size=[new_s])
            cx = (worms["pos_x"][i] + random.uniform(-10, 10)) % WORLD_SIZE
            cy = (worms["pos_y"][i] + random.uniform(-10, 10)) % WORLD_SIZE
            world.spawn(
                "Energy,Heading,Position,WormSize",
                energy=new_e * 0.8,
                heading=random.uniform(0, 6.28),
                pos_x=cx,
                pos_y=cy,
                worm_size=new_s,
            )

    # ── Python: starvation ──
    worms = world.query("Energy", "Position")
    for i in range(len(worms)):
        if worms["energy"][i] <= 0.0:
            world.despawn(worms["entity_id"][i])

    # ── Respawn food ──
    n_new = random.randint(2, 5)
    world.spawn_batch(
        "Nutrition,Position",
        {
            "nutrition": [50.0] * n_new,
            "pos_x": [random.uniform(0, WORLD_SIZE) for _ in range(n_new)],
            "pos_y": [random.uniform(0, WORLD_SIZE) for _ in range(n_new)],
        },
    )

    # ── Track stats ──
    w = len(world.query("Energy", "Position"))
    f = len(world.query("Nutrition", "Position"))
    worm_counts.append(w)
    food_counts.append(f)

    if frame % 5 == 0:
        frames.append(
            (
                world.query("Energy", "Position", "WormSize"),
                world.query("Nutrition", "Position"),
            )
        )
    if frame % 50 == 0:
        print(f"Frame {frame}: {w} worms, {f} food")

## Final State

In [ ]:
worms = world.query("Energy", "Position", "WormSize")
food = world.query("Nutrition", "Position")

fig, ax = plt.subplots(figsize=(8, 8))
if len(food) > 0:
    ax.scatter(
        food["pos_x"], food["pos_y"], s=8, c="limegreen", alpha=0.5, label="Food"
    )
if len(worms) > 0:
    sc = ax.scatter(
        worms["pos_x"],
        worms["pos_y"],
        s=worms["worm_size"].to_numpy() * 30,
        c=worms["energy"].to_numpy(),
        cmap="YlOrRd",
        vmin=0,
        vmax=200,
        edgecolors="black",
        linewidths=0.3,
        label="Worms",
    )
    plt.colorbar(sc, ax=ax, label="Energy")
ax.set_xlim(0, WORLD_SIZE)
ax.set_ylim(0, WORLD_SIZE)
ax.set_aspect("equal")
ax.legend()
ax.set_title(f"Frame {FRAMES} — {len(worms)} worms, {len(food)} food")
plt.tight_layout()
plt.show()

## Population Dynamics

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(worm_counts, color="tab:red", linewidth=1.5, label="Worms")
ax1.set_xlabel("Frame")
ax1.set_ylabel("Worm Count", color="tab:red")

ax2 = ax1.twinx()
ax2.plot(food_counts, color="tab:green", linewidth=1.5, label="Food")
ax2.set_ylabel("Food Count", color="tab:green")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.95))
plt.title("Population Over Time")
plt.tight_layout()
plt.show()

## Energy Distribution

In [ ]:
worms = world.query("Energy", "Position")
if len(worms) > 0:
    energies = worms["energy"].to_numpy()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(energies, bins=30, color="tab:orange", edgecolor="black", alpha=0.7)
    ax.axvline(
        energies.mean(),
        color="red",
        linestyle="--",
        label=f"Mean: {energies.mean():.1f}",
    )
    ax.set_xlabel("Energy")
    ax.set_ylabel("Count")
    ax.set_title("Worm Energy Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("All worms died!")

## Animation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for i, (w_df, f_df) in enumerate(frames):
    ax.clear()
    if len(f_df) > 0:
        ax.scatter(f_df["pos_x"], f_df["pos_y"], s=8, c="limegreen", alpha=0.5)
    if len(w_df) > 0:
        ax.scatter(
            w_df["pos_x"],
            w_df["pos_y"],
            s=w_df["worm_size"].to_numpy() * 30,
            c=w_df["energy"].to_numpy(),
            cmap="YlOrRd",
            vmin=0,
            vmax=200,
            edgecolors="black",
            linewidths=0.3,
        )
    ax.set_xlim(0, WORLD_SIZE)
    ax.set_ylim(0, WORLD_SIZE)
    ax.set_aspect("equal")
    ax.set_title(f"Frame {i * 5} — {len(w_df)} worms, {len(f_df)} food")
    clear_output(wait=True)
    plt.pause(0.05)
    display(fig)

plt.close(fig)

## Explore

- Increase `drain_rate` for more starvation pressure
- Increase `FEED_RADIUS` for easier feeding (worms survive longer)
- Adjust `N_WORMS` vs `N_FOOD` ratio — what happens with 500 worms and 100 food?
- Change fission threshold (`energy > 150`) — lower means more splitting, higher means larger worms
- Increase `speed` — faster worms find food easier but drain energy faster